# ⚙️ Digital Twin of a 3-Phase Induction Motor — Colab Demo

Runs the physics + thermal engine and plots the torque–speed curve, a thermal
warm-up, and a full transient. No Streamlit required.

In [ ]:
# If running in Colab, clone the repo (replace with your GitHub URL)
import os, sys
if not os.path.exists('motor_twin'):
    !git clone https://github.com/YOUR_USERNAME/digital-motor-twin.git
    sys.path.insert(0, 'digital-motor-twin')
!pip -q install plotly numpy pandas

In [ ]:
from motor_twin import InductionMotor, DigitalTwin, Operating, recommend
import plotly.graph_objects as go
import pandas as pd

motor = InductionMotor(v_line=400, frequency=50, poles=4)
print('Synchronous speed:', motor.sync_speed_rpm, 'rpm')
op = motor.operating_point(0.03)
print(f"At 3% slip -> {op['speed_rpm']:.0f} rpm, {op['torque_em']:.1f} N·m, "
      f"{op['efficiency']*100:.0f}% eff, {op['p_out']/1000:.2f} kW")

## Torque–speed characteristic

In [ ]:
sp, tq = motor.torque_speed_curve(200)
fig = go.Figure()
fig.add_scatter(x=sp, y=tq, name='EM torque')
fig.update_layout(title='Torque–speed curve', xaxis_title='speed [RPM]',
                  yaxis_title='torque [N·m]', height=400)
fig.show()

## Transient: start-up + thermal warm-up under load

In [ ]:
twin = DigitalTwin(motor=motor)
twin.reset(t_ambient=25)
states = twin.run(Operating(load_torque=45, airflow_m3h=80), duration=60, dt=0.01)
df = pd.DataFrame(states)

fig = go.Figure()
fig.add_scatter(x=df['time'], y=df['speed_rpm'], name='Speed [RPM]', yaxis='y1')
fig.add_scatter(x=df['time'], y=df['t_winding'], name='Winding °C', yaxis='y2')
fig.update_layout(title='Start-up & thermal response', xaxis_title='time [s]',
                  yaxis=dict(title='RPM'),
                  yaxis2=dict(title='°C', overlaying='y', side='right'),
                  height=420)
fig.show()

print('Final state:')
for k in ['speed_rpm','efficiency','t_winding','p_in','p_out']:
    print(f'  {k}: {df.iloc[-1][k]:.2f}')
for r in recommend(states[-1]):
    print(' -', r['severity'], r['message'])

## Try it
Change the load, cooling airflow or lubrication in `Operating(...)` above and
re-run to see the twin respond. Next milestone: ML failure prediction trained
on the NASA bearing dataset.